# 02. Handoffs (핸드오프 심화)

**핸드오프(Handoff)** 는 에이전트가 특정 작업을 다른 에이전트에게 위임하는 기능입니다.  
핸드오프는 LLM에게 **도구(tool)** 로 표현됩니다.  
예) `Refund Agent`로의 핸드오프 → LLM 도구 이름: `transfer_to_refund_agent`

| 방식 | 예시 | 특징 |
|------|------|------|
| Agent 직접 전달 | `handoffs=[billing_agent]` | 간단, 기본 동작 |
| `handoff()` 함수 | `handoffs=[handoff(billing_agent, on_handoff=...)]` | 콜백, 이름 재정의, 입력 필터 등 고급 옵션 |

**`handoff()` 함수의 주요 파라미터:**

| 파라미터 | 설명 |
|---------|------|
| `agent` | 위임할 대상 에이전트 |
| `tool_name_override` | LLM에게 노출되는 도구 이름 재정의 (기본: `transfer_to_<agent_name>`) |
| `tool_description_override` | 도구 설명 재정의 |
| `on_handoff` | 핸드오프 호출 시 실행되는 콜백 함수 |
| `input_type` | LLM이 핸드오프 시 함께 전달할 데이터 타입 (Pydantic 모델) |
| `input_filter` | 다음 에이전트에게 전달되는 대화 기록 필터링 함수 |
| `is_enabled` | 핸드오프 활성화 여부 (bool 또는 런타임 함수) |

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
Model = "gpt-5-nano"

## 1. handoff() 함수 기본 사용법

Agent 인스턴스를 직접 전달하는 방식과 `handoff()` 함수를 사용하는 방식은 **동일하게 동작**합니다.  
`handoff()` 함수 형태는 추가 옵션이 필요할 때 사용합니다.

In [3]:
from agents import Agent, Runner, handoff

billing_agent = Agent(
    name="Billing agent",
    instructions="당신은 청구 관련 질문을 전문으로 처리합니다.",
    model=Model
)

refund_agent = Agent(
    name="Refund agent",
    instructions="당신은 환불 요청을 전문으로 처리합니다.",
    model=Model
)

triage_agent = Agent(
    name="Triage agent",
    instructions="사용자 요청을 분류하여 적합한 에이전트에게 넘겨주세요.",
    model=Model,
    handoffs=[
        billing_agent,                       # Agent 인스턴스 직접 전달
        handoff(                             # handoff() 함수 — 동일하게 동작
            refund_agent,
            tool_name_override="request_refund",
            tool_description_override="환불 요청이 있을 때 사용",
        ),
    ],
)

result = await Runner.run(triage_agent, input="제 청구서에 오류가 있는 것 같습니다.")
print(result.final_output)

청구서 오류 문제를 Billing 팀에 접수했습니다.

원활한 처리를 위해 아래 정보를 알려주시면 빠르게 조사하고 정정 절차를 진행할 수 있습니다.

필요한 정보
- 청구서/송장 번호
- 계정 식별 정보(계정 ID 또는 등록된 이메일)
- 청구서 발행일 및 청구 기간
- 문제의 구체적 설명(예: 금액 불일치, 중복 청구, 항목 누락 등)
- 증빙 자료(해당 청구서의 스크린샷이나 PDF 등)
- 원하시는 해결 방식(크레딧, 수정 청구서 발행, 환불 등) 또는 구체적 요청
- 연락 가능한 전화번호/이메일(추가 확인 시 필요)

참고
- 일반적으로 Billing 팀은 1-2 영업일 이내에 업데이트를 드립니다.
- 민감한 정보는 안전한 채널로 공유해 주세요. 필요 시 해당 정보를 비공개로 이메일이나 보안 채널로 전달해도 됩니다.

필요한 정보를 이 대화에 바로 남겨주시면 제가 바로 전달 상황을 추적해 드리겠습니다.


## 2. on_handoff 콜백

`on_handoff` 콜백은 핸드오프가 호출되는 순간 실행됩니다.  
데이터 준비, 로깅, 알림 등 사이드 이펙트 처리에 유용합니다.

- `input_type` 없이 사용 시: `def on_handoff(ctx: RunContextWrapper[None])`  
- `input_type` 함께 사용 시: `async def on_handoff(ctx, input_data: MyModel)` (섹션 3 참고)

In [5]:
from agents import Agent, Runner, handoff, RunContextWrapper

def on_billing_handoff(ctx: RunContextWrapper[None]):
    print("[로그] Billing Agent로 핸드오프 발생")

def on_refund_handoff(ctx: RunContextWrapper[None]):
    print("[로그] Refund Agent로 핸드오프 발생")

billing_agent = Agent(
    name="Billing agent",
    instructions="당신은 청구 관련 질문을 전문으로 처리합니다.",
    model=Model
)

refund_agent = Agent(
    name="Refund agent",
    instructions="당신은 환불 요청을 전문으로 처리합니다.",
    model=Model
)

triage_agent = Agent(
    name="Triage agent",
    instructions="사용자 요청을 분류하여 적합한 에이전트에게 넘겨주세요.",
    model=Model,
    handoffs=[
        handoff(billing_agent, on_handoff=on_billing_handoff),
        handoff(refund_agent, on_handoff=on_refund_handoff),
    ],
)

result = await Runner.run(triage_agent, input="환불을 받고 싶습니다.")
print(result.final_output)

[로그] Refund Agent로 핸드오프 발생
환불 처리 담당 에이전트에게 연결해 드리겠습니다. 몇 가지 정보를 확인하면 처리 속도가 빨라집니다.

필요 정보
- 주문번호(또는 주문 식별자)
- 환불 사유(예: 품질 문제, 미수령, 취소 등)
- 환불 방식 선호(원 결제 수단으로 환불, 또는 제3자 쿠폰/포인트 등)
- 환불 요청 시점의 상태(해당 아이템 반품 여부 등)

가능한 범위로 최소한의 정보부터 알려주시면 에이전트가 바로 접수해 드립니다. 필요하신 경우 제가 바로 에이전트와의 상담을 시작합니다.


## 3. Handoff 입력 데이터 (input_type)

`input_type`을 사용하면 LLM이 핸드오프 시 **구조화된 데이터를 함께 전달**하도록 할 수 있습니다.  
예를 들어, 에스컬레이션 이유를 함께 전달받아 로깅하거나 처리에 활용할 수 있습니다.

- `input_type`이 있을 때 `on_handoff`는 `async def`로 정의하고 두 번째 인자로 `input_data`를 받습니다.

In [6]:
from pydantic import BaseModel
from agents import Agent, Runner, handoff, RunContextWrapper

class EscalationData(BaseModel):
    reason: str

async def on_escalation(ctx: RunContextWrapper[None], input_data: EscalationData):
    print(f"[에스컬레이션] 이유: {input_data.reason}")

escalation_agent = Agent(
    name="Escalation agent",
    instructions="당신은 복잡한 문제를 처리하는 전문 상담사입니다.",
    model=Model
)

support_agent = Agent(
    name="Support agent",
    instructions=(
        "일반 고객 지원 요청을 처리하세요. "
        "복잡하거나 민감한 문제는 에스컬레이션 에이전트에게 이유와 함께 넘겨주세요."
    ),
    model=Model,
    handoffs=[
        handoff(
            escalation_agent,
            on_handoff=on_escalation,
            input_type=EscalationData,
        ),
    ],
)

result = await Runner.run(support_agent, input="제 계정이 해킹당한 것 같습니다. 즉시 도움이 필요합니다!")
print(result.final_output)

[에스컬레이션] 이유: 계정 해킹 의심으로 인한 긴급 보안 조치 필요. 사용자의 신원 확인 및 비밀번호 재설정, 활성 세션 종료, 2FA 점검 등 신속한 조치와 조치 진행 상황 안내 필요.
도와드리겠습니다. 계정 해킹은 긴급한 보안 이슈이므로 이미 에스컬레이션 팀에 바로 넘겼습니다. 에이전트가 곧 연락드릴 예정이며, 아래의 즉시 조치도 함께 따라주시면 빠르게 해결될 가능성이 큽니다.

지금 바로 가능한 보안 조치 (권장)
- 가능하면 보안이 확인된 다른 기기에서 로그인하고, 이 기기에서는 현재 세션에서 로그아웃합니다. 필요하면 모든 활성 세션을 종료하세요.
- 비밀번호를 강력하고 고유한 새 비밀번호로 변경합니다. 가능하면 해당 서비스의 비밀번호를 이메일/다른 서비스와도 다른 것으로 만드세요.
- 2단계 인증(2FA)을 즉시 활성화합니다. 이미 활성화되어 있다면 설정을 재확인하고, 인증 앱/메시지 기반 2FA가 정상 작동하는지 확인하세요.
- 이메일 계정의 보안도 강화합니다. 이메일 계정이 해킹에 취약해지면 다른 서비스도 위험해질 수 있습니다. 이메일 비밀번호도 새로 설정하고 2FA를 사용하세요.
- 결제 정보나 구독 정보가 연결되어 있다면 모니터링하고, 의심스러운 결제나 변경이 있으면 즉시 서비스에 보고합니다.
- 의심스러운 활동의 기록을 남깁니다(로그인 시각, IP 대략 위치, 사용 기기, 변경된 설정 등).

에이전트가 요청할 수 있는 정보(빠른 처리를 위해 미리 준비해두면 좋습니다)
- 계정 이름 또는 로그인 이메일 주소
- 서비스 종류(예: 이메일, 소셜 네트워크, 은행/결제 서비스 등)
- 최근 정상 로그인이 있었던 시점
- 의심되거나 확인되지 않은 활동의 구체적 내역(로그인 시도, 위치, 변경된 설정 등)
- 현재 활성화된 2FA 여부와 사용 중인 인증 방식
- 현재 연결된 기기 목록 및 최근 로그아웃 여부
- 변경되거나 의심스러운 비밀번호 변경 여부
- 결제 정보나 구독 상태의 변경 여부

주의 사항
- 이 채널에 민감한 정보를 입력하기 전에는 반

## 4. Input Filter (입력 필터)

핸드오프가 발생하면 새로운 에이전트는 기존 대화 기록 전체를 받습니다.  
`input_filter`를 사용하면 다음 에이전트에게 전달되는 **대화 기록을 가공**할 수 있습니다.

`agents.extensions.handoff_filters`에는 자주 사용되는 내장 필터가 제공됩니다:

| 필터 | 설명 |
|------|------|
| `remove_all_tools` | 대화 기록에서 모든 도구 호출/결과 제거 |

In [7]:
from agents import Agent, Runner, handoff
from agents.extensions import handoff_filters

faq_agent = Agent(
    name="FAQ agent",
    instructions="자주 묻는 질문에 답변합니다.",
    model=Model
)

order_agent = Agent(
    name="Order agent",
    instructions="주문 상태를 확인하고 주문 관련 문의를 처리합니다.",
    model=Model
)

main_agent = Agent(
    name="Main agent",
    instructions="고객 문의를 처리하세요. 필요 시 적절한 전문 에이전트에게 넘겨주세요.",
    model=Model,
    handoffs=[
        handoff(
            faq_agent,
            input_filter=handoff_filters.remove_all_tools,
        ),
        order_agent,
    ],
)

result = await Runner.run(main_agent, input="배송 추적은 어떻게 하나요?")
print(result.final_output)

다음과 같이 배송 추적을 확인할 수 있습니다.

- 배송 추적 번호 찾기: 주문 확인 메일이나 앱에서 배송 추적 번호를 확인합니다.
- 택배사 사이트 이용: 해당 추적 번호를 택배사 공식 웹사이트나 앱에 입력하여 위치와 상태를 확인합니다.
- 우리 사이트에서 확인: 앱 또는 웹에서 “주문 관리” > “배송 추적”으로도 현재 상태를 볼 수 있습니다.
- 정보 확인 포인트: 현재 위치, 남은 예상 도착일, 지연 여부 등을 확인합니다.

도움이 더 필요하시면 특정 주문의 현황을 바로 확인해 드릴 수 있습니다. 주문 번호를 알려주시면 배송 추적을 확인해 드리거나, 필요 시 배송 상담원으로 연결해 드리겠습니다.


## 5. 권장 프롬프트 (RECOMMENDED_PROMPT_PREFIX)

LLM이 핸드오프를 올바르게 이해하고 활용하려면 관련 정보를 프롬프트에 포함시키는 것이 좋습니다.  
`agents.extensions.handoff_prompt`에서 권장 프롬프트를 제공합니다:

| 방법 | 설명 |
|------|------|
| `RECOMMENDED_PROMPT_PREFIX` | 프롬프트 앞에 직접 붙여 사용하는 문자열 상수 |
| `prompt_with_handoff_instructions(prompt)` | 기존 프롬프트에 권장 내용을 자동으로 추가하는 함수 |

In [8]:
from agents import Agent, Runner, handoff
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX, prompt_with_handoff_instructions

billing_agent = Agent(
    name="Billing agent",
    instructions=f"""{RECOMMENDED_PROMPT_PREFIX}
당신은 청구 및 결제 관련 질문을 처리하는 전문가입니다.""",
    model=Model
)

refund_agent = Agent(
    name="Refund agent",
    instructions=prompt_with_handoff_instructions("당신은 환불 처리 전문가입니다."),
    model=Model
)

triage_agent = Agent(
    name="Triage agent",
    instructions=prompt_with_handoff_instructions(
        "사용자 요청을 분석하여 청구 질문은 Billing agent에게, 환불 요청은 Refund agent에게 넘겨주세요."
    ),
    model=Model,
    handoffs=[billing_agent, refund_agent],
)

result = await Runner.run(triage_agent, input="지난달 청구 금액이 잘못된 것 같습니다.")
print(result.final_output)

청구 관련 문의를 확인하고 해결해 드리겠습니다.

가능하신 한 구체적인 정보를 알려주시면 빠르게 확인에 도움이 됩니다.
- 문제의 청구 기간(예: 지난 달의 시작일 ~ 종료일)
- 의심되는 금액 또는 항목명
- 청구서 번호(있다면)
- 화면에 보이는 금액과 차이가 나는지 여부

추가로 화면 스크린샷이나 파일이 있다면 첨부해 주세요. 확인 후 Billing 팀에서 재검토하고 연락드리겠습니다.


## 6. 종합 예제 - 고객 지원 시스템

여러 핸드오프 기능을 조합한 고객 지원 시스템입니다.

```
사용자
  └─→ Triage Agent (분류)
        ├─→ Order Agent    (주문 조회)
        ├─→ Refund Agent   (환불 처리, on_handoff + input_type)
        └─→ FAQ Agent      (일반 문의, input_filter 적용)
```

In [9]:
from pydantic import BaseModel
from agents import Agent, Runner, handoff, RunContextWrapper
from agents.extensions import handoff_filters
from agents.extensions.handoff_prompt import prompt_with_handoff_instructions

class RefundRequest(BaseModel):
    order_id: str
    reason: str

async def on_refund_handoff(ctx: RunContextWrapper[None], input_data: RefundRequest):
    print(f"[환불 요청 접수] 주문번호: {input_data.order_id}, 사유: {input_data.reason}")

order_agent = Agent(
    name="Order Agent",
    instructions=prompt_with_handoff_instructions(
        "주문 상태 및 배송 관련 문의를 처리합니다. 주문번호를 확인하고 현황을 안내하세요."
    ),
    model=Model
)

refund_agent = Agent(
    name="Refund Agent",
    instructions=prompt_with_handoff_instructions(
        "환불 요청을 처리합니다. 고객에게 환불 절차와 소요 시간을 안내하세요."
    ),
    model=Model
)

faq_agent = Agent(
    name="FAQ Agent",
    instructions=prompt_with_handoff_instructions(
        "자주 묻는 질문에 답변합니다. 배송, 반품 정책, 결제 방법 등 일반적인 문의를 처리하세요."
    ),
    model=Model
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=prompt_with_handoff_instructions(
        "고객 문의를 분류하세요:\n"
        "- 주문/배송 조회 → Order Agent\n"
        "- 환불 요청 → Refund Agent (주문번호와 사유 필요)\n"
        "- 일반 문의/FAQ → FAQ Agent"
    ),
    model=Model,
    handoffs=[
        order_agent,
        handoff(
            refund_agent,
            on_handoff=on_refund_handoff,
            input_type=RefundRequest,
        ),
        handoff(
            faq_agent,
            input_filter=handoff_filters.remove_all_tools,
        ),
    ],
)

In [10]:
print("=== 테스트 1: 주문 조회 ===")
result = await Runner.run(triage_agent, "주문번호 ORD-1234의 배송 현황을 알고 싶습니다.")
print(result.final_output)

=== 테스트 1: 주문 조회 ===
주문 ORD-1234의 배송 현황을 확인하고 있습니다. 확인되는 대로 바로 알려드리겠습니다. 잠시만 기다려 주세요.


In [11]:
print("=== 테스트 2: 환불 요청 ===")
result = await Runner.run(triage_agent, "주문번호 ORD-5678 상품을 환불하고 싶습니다. 사이즈가 맞지 않아서요.")
print(result.final_output)

=== 테스트 2: 환불 요청 ===
[환불 요청 접수] 주문번호: ORD-5678, 사유: 사이즈가 맞지 않아 환불 요청
환불 요청이 접수되었습니다.
- 주문번호: ORD-5678
- 사유: 사이즈가 맞지 않아

처리 기간 안내:
- 일반적으로 영업일 기준 5-7일 정도 소요됩니다.
- 환불은 사용하신 결제수단으로 처리되며, 처리 상태는 등록된 연락처로 안내드립니다.

다른 도움이 필요하시면 말씀해 주세요. 교환을 원하신다면 그 역시 도와드리겠습니다.


In [12]:
print("=== 테스트 3: 일반 문의 ===")
result = await Runner.run(triage_agent, "반품 정책이 어떻게 되나요?")
print(result.final_output)

=== 테스트 3: 일반 문의 ===
일반적인 반품 정책은 구매처에 따라 다르지만, 대부분 아래와 같은 요소로 구성됩니다. 참고용으로 확인해 보시고, 구체적인 정책이 필요하시면 구매처를 알려 주세요.

주요 내용
- 반품 기간: 보통 수령일로부터 30일 이내 반품 가능이 일반적입니다. 다만 상품군이나 프로모션에 따라 기간이 다를 수 있습니다(예: 7–14일).
- 상태와 포장: 사용하지 않은 새 상품이어야 하며, 원래 포장 및 태그를 함께 반품해야 하는 경우가 많습니다.
- 제외 품목: 맞춤 제작, 세일 제외 품목, 위생상 이유로 반품이 제한되는 상품(일부 뷰티/위생 용품 등), 디지털 콘텐츠 다운로드 등은 예외로 적용될 수 있습니다.
- 반품 방법: 일반적으로 반품 요청(또는 반품 승인(RMA) 받기) 후 반품 배송 방법이 안내됩니다. 무료 반품 라벨이 제공되기도 하고, 배송비를 고객이 부담해야 하는 경우도 있습니다.
- 환불/교환: 반품 도착 후 상품 검사에 따라 환불이나 교환이 처리됩니다. 환불은 원 결제 수단으로 이루어지는 것이 일반적이며, 환불 소요 기간은 보통 3–10영업일 정도입니다. 일부 결제 수단은 처리 시간이 다를 수 있습니다.
- 상태 점검 시점: 반품 도착 후 검사 없이 바로 처리되기도 하고, 반품 사유를 확인한 뒤 처리되기도 합니다.

고객님이 직접 확인하면 좋은 정보
- 주문처/판매처 이름(브랜드나 온라인 몰)
- 주문 번호
- 반품 사유(사이즈 문제, 품질 문제, 오배송 등)
- 반품하려는 상품의 상태와 포장 여부
- 원하시는 처리 방식(환불, 교환 중)

원하시면 구매처를 알려 주시거나 주문 번호를 주시면 해당 반품 정책에 맞춰 자세한 절차를 구체적으로 안내해 드리겠습니다.


### 실습 문제

아래 요구사항에 맞는 **여행 예약 지원 시스템**을 구현하세요.

**에이전트 구성:**

1. **Triage Agent**: 사용자 요청을 분류하여 적절한 에이전트에 핸드오프
2. **Flight Agent**: 항공편 예약 및 조회 처리
3. **Hotel Agent**: 호텔 예약 및 조회 처리
4. **Cancellation Agent**: 예약 취소 처리 (`on_handoff` 콜백으로 취소 정보 로깅)

**요구사항:**
- `Cancellation Agent`로 핸드오프 시 `input_type`으로 예약 번호(`booking_id: str`)와 취소 사유(`reason: str`)를 전달받아 출력
- 모든 에이전트에 `prompt_with_handoff_instructions` 적용
- `Flight Agent`, `Hotel Agent`는 `handoff()` 함수 형태로 지정

**테스트 입력:**
- `"서울-제주 항공편을 예약하고 싶습니다."` → Flight Agent 응답
- `"제주도 호텔을 3박 예약하려고 합니다."` → Hotel Agent 응답
- `"예약번호 BK-999 항공편을 취소하고 싶어요. 일정이 바뀌어서요."` → Cancellation Agent 핸드오프 + 콜백 출력